Esse Notebook  realiza o primeiro tratamento na base_dados_cadastrais, principla obejetivo é identificar como os dados e variáveis estão na base e  realizar o primeiro ETL para a camada Silver



Instaladando as dependências e bibliotecas

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:


# Inicializando a sessão
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Camada Silver") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

# Verificando se funcionou
print("Sessão Spark criada com sucesso!")
spark

Sessão Spark criada com sucesso!


In [ ]:
from pyspark.sql.functions import input_file_name, split, col, lit, regexp_replace
from datetime import datetime, timedelta
from decimal import Decimal
from pyspark.sql.functions import col, count, when, isnull, isnan, countDistinct, round, variance, stddev
from typing import Dict, List, Tuple, Optional
from pyspark.sql.types import NumericType, StringType
from pyspark.sql import functions as F
import os
import time
from datetime import datetime

In [ ]:
#Criar uma função log para registrar a data
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') +' >>>'
dt_proc = datetime.now().strftime('%Y%m%d%H%M%S')
current_date = datetime.now().strftime('%Y%m%d')

In [ ]:
log()

'2026-01-01 21:51:24 >>>'

Carregando a base de dados e fazendo as alterações proposta na analise posteriormente

In [ ]:
# Caso for utlizar o colab retirar os #
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
#Carregando os dados da tabela
df_base_dados_cadastrais = spark.read.parquet("/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_dados_cadastrais", header=True, inferSchema=True)
df_base_dados_cadastrais.createOrReplaceTempView("df_base_dados_cadastrais")

In [ ]:
# Mostrando fomato do banco de dados
df_base_dados_cadastrais.show()

+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+-------+------+------+------+------+----------+----------+------+------+------+------+----------+--------+------+------------+------------+-------------+----------+--------------------+-------------+
|    NUM_CPF| SAFRA|FLAG_INSTALACAO| FPD|PROD|flag_mig2|STATUSRF|DATADENASCIMENTO|var_03|var_02|var_04|var_05|var_06| var_07|var_08|var_09|var_10|var_11|    var_12|    var_13|var_14|var_15|var_16|var_17|    var_18|  var_19|var_20|      var_21|      var_22|       var_23|    var_24|              var_25|CEP_3_digitos|
+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+-------+------+------+------+------+----------+----------+------+------+------+------+----------+--------+------+------------+------------+-------------+----------+--------------------+-------------+
|77789989YZZ|202503|              1|   0| CMV|   

In [ ]:
#verificando o numero de linhas
df_base_dados_cadastrais.count()

3900378

In [ ]:
# Verificando o esquema da tabela
df_base_dados_cadastrais.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: string (nullable = true)
 |-- FLAG_INSTALACAO: string (nullable = true)
 |-- FPD: string (nullable = true)
 |-- PROD: string (nullable = true)
 |-- flag_mig2: string (nullable = true)
 |-- STATUSRF: string (nullable = true)
 |-- DATADENASCIMENTO: string (nullable = true)
 |-- var_03: string (nullable = true)
 |-- var_02: string (nullable = true)
 |-- var_04: string (nullable = true)
 |-- var_05: string (nullable = true)
 |-- var_06: string (nullable = true)
 |-- var_07: string (nullable = true)
 |-- var_08: string (nullable = true)
 |-- var_09: string (nullable = true)
 |-- var_10: string (nullable = true)
 |-- var_11: string (nullable = true)
 |-- var_12: string (nullable = true)
 |-- var_13: string (nullable = true)
 |-- var_14: string (nullable = true)
 |-- var_15: string (nullable = true)
 |-- var_16: string (nullable = true)
 |-- var_17: string (nullable = true)
 |-- var_18: string (nullable = true)
 |-- var_19: string (nulla

In [ ]:
Data_Safra = spark.sql("""
    SELECT
        MIN(SAFRA) as inicio,
        MAX(SAFRA) as fim
    FROM df_base_dados_cadastrais
""").collect()

# 3. Extrair os valores do resultado (primeira linha, colunas 0 e 1)
inicio_safra = Data_Safra[0]['inicio']
fim_safra = Data_Safra[0]['fim']

# 4. Exibir usando função log() matem registro de data quando ocorrer alterações nos dados
print(f"{log()} Inicio da safra: {inicio_safra}")
print(f"{log()} Fim da safra: {fim_safra}")

2026-01-01 21:51:41 >>> Inicio da safra: 202410
2026-01-01 21:51:41 >>> Fim da safra: 202503


Para o processo de ETL:
Colocar o datatype correto

Nome: Safra Transformar coluna em INT Criar duas nova colunas: Ano e Mes

Inicio da safra:  202410

Fim da safra:  202503

In [ ]:
df_silver_base_dados_cadastrais = spark.sql(f"""
    SELECT

        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        '{dt_proc}' AS DATA_PROCESSAMENTO,

        -- ETL de Safra: Transformação para INT e criação de Ano/Mês
        CAST(SAFRA AS INT) AS SAFRA,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS ANO,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS MES,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS FLAG_INSTALACAO,
        CAST(FPD AS BOOLEAN) AS FPD,
        CAST(PROD AS STRING) AS PROD,
        CAST(flag_mig2 AS STRING) AS flag_mig2,
        CAST(STATUSRF AS STRING) AS STATUSRF,

        -- Correção de Datas (usando to_date com o formato específico)
        to_date(DATADENASCIMENTO, 'dd/MM/yyyy') AS DATA_DE_NASCIMENTO,
        to_date(var_12, 'dd/MM/yyyy') AS var_12,

        CAST(var_02 AS INT) AS var_02,
        CAST(var_03 AS INT) AS var_03,
        CAST(var_04 AS INT) AS var_04,
        CAST(var_05 AS INT) AS var_05,
        CAST(var_06 AS INT) AS var_06,
        CAST(var_07 AS FLOAT) AS var_07,
        CAST(var_08 AS INT) AS var_08,
        CAST(var_09 AS INT) AS var_09,
        CAST(var_10 AS STRING) AS Profissao,
        CAST(var_11 AS FLOAT) AS var_11,

        -- var_13 mantida como string por ser mista
        CAST(var_13 AS STRING) AS var_13,

        CAST(var_14 AS INT) AS var_14,
        CAST(var_15 AS STRING) AS Estado,
        CAST(var_16 AS INT) AS var_16,
        CAST(var_17 AS INT) AS var_17,
        CAST(var_18 AS STRING) AS var_18,
        CAST(var_19 AS STRING) AS var_19,
        CAST(var_20 AS STRING) AS var_20,
        CAST(var_21 AS STRING) AS var_21,
        CAST(var_22 AS STRING) AS Cargo,
        CAST(var_23 AS STRING) AS var_23,
        CAST(var_24 AS STRING) AS var_24,
        CAST(var_25 AS STRING) AS Tipo_de_Auxilio,
        CAST(CEP_3_digitos AS STRING) AS CEP_3_digitos

    FROM df_base_dados_cadastrais
    WHERE CAST(SAFRA AS INT) BETWEEN 202410 AND 202503
""")

In [ ]:
df_silver_base_dados_cadastrais.show(20, False)

+-----------+------------------+------+----+---+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+-------+------+------+---------+------+----------+------+------+------+------+----------+--------+------+------------+------------+-------------+----------+---------------------------------------------+-------------+
|NUM_CPF    |DATA_PROCESSAMENTO|SAFRA |ANO |MES|FLAG_INSTALACAO|FPD  |PROD|flag_mig2|STATUSRF|DATA_DE_NASCIMENTO|var_12    |var_02|var_03|var_04|var_05|var_06|var_07 |var_08|var_09|Profissao|var_11|var_13    |var_14|Estado|var_16|var_17|var_18    |var_19  |var_20|var_21      |Cargo       |var_23       |var_24    |Tipo_de_Auxilio                              |CEP_3_digitos|
+-----------+------------------+------+----+---+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+-------+------+------+---------+------+----------+------+------+------+------+----------+

In [ ]:
df_silver_base_dados_cadastrais.count()

3900378

In [ ]:
# Salvar a tabela na camada silver
df_silver_base_dados_cadastrais.write.mode('append').parquet('/content/gdrive/MyDrive/Raw Hackathon PoD 2025/Silver Hackathon 2025')


In [ ]:
# verificar se a tabela foi salva
df_silver_base_dados_cadastrais = spark.read.parquet("/content/gdrive/MyDrive/Raw Hackathon PoD 2025/Silver Hackathon 2025")


In [ ]:
df_silver_base_dados_cadastrais.show()

+-----------+------------------+------+----+---+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+-------+------+------+---------+------+----------+------+------+------+------+----------+--------+------+------------+------------+-------------+----------+--------------------+-------------+
|    NUM_CPF|DATA_PROCESSAMENTO| SAFRA| ANO|MES|FLAG_INSTALACAO|  FPD|PROD|flag_mig2|STATUSRF|DATA_DE_NASCIMENTO|    var_12|var_02|var_03|var_04|var_05|var_06| var_07|var_08|var_09|Profissao|var_11|    var_13|var_14|Estado|var_16|var_17|    var_18|  var_19|var_20|      var_21|       Cargo|       var_23|    var_24|     Tipo_de_Auxilio|CEP_3_digitos|
+-----------+------------------+------+----+---+---------------+-----+----+---------+--------+------------------+----------+------+------+------+------+------+-------+------+------+---------+------+----------+------+------+------+------+----------+--------+------+------------+------------+--------